# Base de Données - Portail d'Emploi

## Créer une Base de Données SQLite pour un Portail d'Emploi

**Objectifs:**
- Créer 3 tables relationnelles (ENTREPRISE, OFFRE, PROFIL_CANDIDAT)
- Insérer des données réalistes (32 enregistrements)
- Exécuter des requêtes SQL variées (débutant à avancé)
- Analyser les données avec des agrégations

**Durée:** 60-90 minutes

---

## 1. Configuration et Imports

Importons les bibliothèques nécessaires et créons la connexion SQLite.

In [ ]:
import sqlite3
import pandas as pd
from datetime import datetime

# Si la cellule est relancée, on ferme proprement l'ancienne connexion
try:
    conn.close()
except Exception:
    pass

# Base de données en mémoire (repart de zéro à chaque exécution de cette cellule)
conn = sqlite3.connect(':memory:')
conn.execute("PRAGMA foreign_keys = ON")
cursor = conn.cursor()

print("✓ Connexion à SQLite établie")
print("✓ Base de données créée en mémoire")
print(f"✓ Date/Heure: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2. Création des Tables

Créons les 3 tables avec leurs contraintes et clés étrangères.

In [ ]:
# Supprimer les tables si elles existent (permet de relancer la cellule sans erreur)
cursor.execute("DROP TABLE IF EXISTS offre")
cursor.execute("DROP TABLE IF EXISTS profil_candidat")
cursor.execute("DROP TABLE IF EXISTS entreprise")

# Table ENTREPRISE
cursor.execute('''
    CREATE TABLE entreprise (
        entreprise_id INTEGER PRIMARY KEY AUTOINCREMENT,
        nom_entreprise VARCHAR(100) NOT NULL UNIQUE,
        secteur VARCHAR(50) NOT NULL,
        localisation VARCHAR(100) NOT NULL,
        nombre_employes INTEGER CHECK (nombre_employes >= 1),
        date_creation DATE DEFAULT CURRENT_DATE,
        site_web VARCHAR(100) UNIQUE
    )
''')

# Table OFFRE
cursor.execute('''
    CREATE TABLE offre (
        offre_id INTEGER PRIMARY KEY AUTOINCREMENT,
        entreprise_id INTEGER NOT NULL,
        titre_poste VARCHAR(100) NOT NULL,
        description TEXT,
        salaire_min DECIMAL(10,2) CHECK (salaire_min >= 0),
        salaire_max DECIMAL(10,2) CHECK (salaire_max >= 0),
        niveau_experience VARCHAR(50) NOT NULL,
        competences_requises VARCHAR(200),
        date_publication DATE DEFAULT CURRENT_DATE,
        date_limite DATE,
        type_contrat VARCHAR(50) NOT NULL,
        statut VARCHAR(50) DEFAULT 'Ouvert',
        FOREIGN KEY (entreprise_id) REFERENCES entreprise(entreprise_id)
    )
''')

# Table PROFIL_CANDIDAT
cursor.execute('''
    CREATE TABLE profil_candidat (
        candidat_id INTEGER PRIMARY KEY AUTOINCREMENT,
        nom_candidat VARCHAR(100) NOT NULL,
        email VARCHAR(100) UNIQUE NOT NULL,
        telephone VARCHAR(20),
        localisation VARCHAR(100) NOT NULL,
        date_naissance DATE,
        niveau_etudes VARCHAR(50) NOT NULL,
        annees_experience INTEGER CHECK (annees_experience >= 0),
        specialite VARCHAR(100),
        competences VARCHAR(200),
        cv_telecharge DATE,
        statut_candidat VARCHAR(50) DEFAULT 'Actif',
        date_inscription DATE DEFAULT CURRENT_DATE
    )
''')

conn.commit()
print("✓ Table ENTREPRISE créée")
print("✓ Table OFFRE créée")
print("✓ Table PROFIL_CANDIDAT créée")

## 3. Insertion des Données

Insérons les données réalistes (5 entreprises, 12 offres, 15 candidats).

In [ ]:
# Vider la table avant insertion (évite les doublons si la cellule est relancée)
cursor.execute("DELETE FROM entreprise")
cursor.execute("DELETE FROM sqlite_sequence WHERE name = 'entreprise'")

# Insérer les entreprises
entreprises = [
    ('TechVision Maroc', 'Tech', 'Casablanca', 150, '2015-03-10', 'www.techvision.ma'),
    ('FinanceHub', 'Finance', 'Rabat', 85, '2018-05-20', 'www.financehub.ma'),
    ('MediCare Plus', 'Santé', 'Fes', 200, '2010-01-15', 'www.medicareplus.ma'),
    ('RetailMax', 'Commerce', 'Marrakech', 320, '2005-06-30', 'www.retailmax.ma'),
    ('EduTech Solutions', 'Éducation', 'Casablanca', 50, '2020-09-08', 'www.edutech.ma')
]

cursor.executemany('''
    INSERT INTO entreprise (nom_entreprise, secteur, localisation, nombre_employes, date_creation, site_web)
    VALUES (?, ?, ?, ?, ?, ?)
''', entreprises)

print(f"✓ {len(entreprises)} entreprises insérées")
conn.commit()

In [ ]:
# Vider la table avant insertion (évite les doublons si la cellule est relancée)
cursor.execute("DELETE FROM offre")
cursor.execute("DELETE FROM sqlite_sequence WHERE name = 'offre'")

# Insérer les offres
offres = [
    (1, 'Développeur Python', 'Cherchons dev Python expérimenté', 25000, 35000, 'Confirmé', 'Python;Django;SQL', '2025-10-01', '2025-10-31', 'CDI', 'Ouvert'),
    (1, 'Junior Developer', 'Jeune diplômé bienvenu', 15000, 20000, 'Débutant', 'Python;HTML;CSS', '2025-10-02', '2025-10-15', 'CDI', 'Ouvert'),
    (2, 'Analyste Financier', 'Analyse de données financières', 30000, 40000, 'Expert', 'Excel;SQL;Power BI', '2025-09-15', '2025-10-20', 'CDI', 'Ouvert'),
    (2, 'Assistant Comptable', 'Support comptable', 12000, 16000, 'Débutant', 'Excel;Comptabilité', '2025-10-05', '2025-10-25', 'CDD', 'Ouvert'),
    (3, 'Infirmier Spécialisé', 'Unité de soins', 18000, 24000, 'Confirmé', 'Nursing;Diagnostique', '2025-09-20', '2025-10-15', 'CDI', 'Fermé'),
    (3, 'Aide-Soignant', 'Support médical', 10000, 13000, 'Débutant', 'Premiers secours', '2025-10-03', '2025-10-31', 'CDI', 'Ouvert'),
    (4, 'Chef de Rayon', 'Gestion rayon retail', 16000, 22000, 'Confirmé', 'Management;Retail', '2025-10-01', '2025-10-20', 'CDI', 'Ouvert'),
    (4, 'Vendeur', 'Accueil clients', 8000, 12000, 'Débutant', 'Vente;Communication', '2025-10-02', '2025-11-05', 'CDI', 'Pourvu'),
    (5, 'Responsable Pédagogique', 'Gestion cursus', 20000, 28000, 'Confirmé', 'Pédagogie;Gestion', '2025-10-01', '2025-10-31', 'CDI', 'Ouvert'),
    (5, 'Formateur IT', 'Cours informatique', 18000, 25000, 'Confirmé', 'IT;Pédagogie', '2025-10-04', '2025-10-31', 'CDD', 'Ouvert'),
    (1, 'Data Analyst', 'Analyse de données', 28000, 38000, 'Expert', 'Python;SQL;Tableau', '2025-09-25', '2025-10-20', 'CDI', 'Ouvert'),
    (2, 'Stagiaire Finance', 'Stage 6 mois', 6000, 8000, 'Débutant', 'Excel;Bases finance', '2025-10-10', '2025-11-10', 'Stage', 'Ouvert')
]

cursor.executemany('''
    INSERT INTO offre (entreprise_id, titre_poste, description, salaire_min, salaire_max,
                       niveau_experience, competences_requises, date_publication, date_limite,
                       type_contrat, statut)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
''', offres)

print(f"✓ {len(offres)} offres insérées")
conn.commit()

In [ ]:
# Vider la table avant insertion (évite les doublons si la cellule est relancée)
cursor.execute("DELETE FROM profil_candidat")
cursor.execute("DELETE FROM sqlite_sequence WHERE name = 'profil_candidat'")

# Insérer les candidats
candidats = [
    ('Ahmed Bennani', 'ahmed.b@email.com', '0612345678', 'Casablanca', '1992-05-15', 'Master', 5, 'Développement web', 'Python;Django;JavaScript', '2025-09-20', 'Actif', '2025-01-10'),
    ('Fatima Zahra Alaoui', 'fatima.z@email.com', '0698765432', 'Fes', '1995-08-22', 'Licence', 3, 'Data Science', 'Python;SQL;Tableau', '2025-09-18', 'Actif', '2025-02-15'),
    ('Mohammed Karim', 'm.karim@email.com', '0645670890', 'Rabat', '1998-11-30', 'Licence', 1, 'Infirmerie', 'Nursing;Premiers secours', '2025-09-25', 'Actif', '2025-03-01'),
    ('Laila Saada', 'l.saada@email.com', '0632145678', 'Marrakech', '1990-03-10', 'Master', 8, 'Finance', 'Excel;Power BI;SQL', '2025-09-22', 'Actif', '2025-04-12'),
    ('Youssef Tahir', 'y.tahir@email.com', '0656789012', 'Agadir', '1999-07-05', 'Bac', 0, 'Vente-Commerce', 'Vente;Communication', '2025-09-28', 'Actif', '2025-05-20'),
    ('Nadia Hassan', 'nadia.h@email.com', '0634567890', 'Casablanca', '1993-02-18', 'Master', 6, 'Gestion RH', 'Management;Pédagogie', '2025-09-20', 'Actif', '2025-06-10'),
    ('Ibrahim Moussaoui', 'ibrahim.m@email.com', '0667890123', 'Rabat', '1996-09-12', 'Licence', 2, 'Informatique', 'Python;Java;HTML', '2025-09-19', 'Actif', '2025-07-05'),
    ('Salima Bouazza', 'salima.b@email.com', '0623456789', 'Fes', '1994-12-25', 'Licence', 4, 'Infirmerie', 'Nursing;Diagnostique', '2025-09-24', 'Actif', '2025-08-15'),
    ('Khalid Amrani', 'khalid.a@email.com', '0645678901', 'Marrakech', '1997-06-08', 'Licence', 2, 'Comptabilité', 'Excel;Comptabilité', '2025-09-21', 'Inactif', '2025-09-01'),
    ('Zainab Bennani', 'zainab.b@email.com', '0612398765', 'Casablanca', '1992-04-30', 'Master', 7, 'Développement', 'Python;SQL;JavaScript', '2025-09-26', 'Actif', '2025-09-10'),
    ('Marwan Leblanc', 'marwan.l@email.com', '0656712345', 'Rabat', '1995-01-14', 'Licence', 3, 'IT', 'Java;Python;SQL', '2025-09-20', 'Actif', '2025-09-15'),
    ('Aisha Farraj', 'aisha.f@email.com', '0634598765', 'Casablanca', '1991-10-22', 'Master', 9, 'Pédagogie', 'Pédagogie;IT;Gestion', '2025-09-25', 'Actif', '2025-09-20'),
    ('Tariq El Fassi', 'tariq.ef@email.com', '0667543210', 'Fes', '1998-08-17', 'Licence', 1, 'Santé', 'Premiers secours;Nursing', '2025-09-23', 'Actif', '2025-09-22'),
    ('Sara Belkadi', 'sara.b@email.com', '0623784590', 'Marrakech', '1993-05-11', 'Licence', 4, 'Retail', 'Management;Vente', '2025-09-27', 'Bloqué', '2025-09-25'),
    ('Rashid Mediane', 'rashid.m@email.com', '0645932187', 'Agadir', '1994-11-09', 'Licence', 5, 'Développement', 'Python;Django', '2025-09-22', 'Actif', '2025-09-28')
]

cursor.executemany('''
    INSERT INTO profil_candidat (nom_candidat, email, telephone, localisation, date_naissance,
                                niveau_etudes, annees_experience, specialite, competences,
                                cv_telecharge, statut_candidat, date_inscription)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
''', candidats)

conn.commit()
print(f"✓ {len(candidats)} candidats insérés")
print(f"\n✓ Total: {len(entreprises)} + {len(offres)} + {len(candidats)} = {len(entreprises) + len(offres) + len(candidats)} enregistrements")

## 4. Vérification des Données

Vérifions que les données ont été correctement insérées.

In [ ]:
# Vérifier les entreprises
df_entreprises = pd.read_sql_query("SELECT * FROM entreprise", conn)
print("=== ENTREPRISES ===")
print(df_entreprises)
print(f"\nNombre d'entreprises: {len(df_entreprises)}")

In [ ]:
# Vérifier les offres
df_offres = pd.read_sql_query("SELECT * FROM offre", conn)
print("=== OFFRES ===")
print(df_offres[['offre_id', 'titre_poste', 'salaire_min', 'salaire_max', 'type_contrat', 'statut']])
print(f"\nNombre d'offres: {len(df_offres)}")

In [ ]:
# Vérifier les candidats
df_candidats = pd.read_sql_query("SELECT * FROM profil_candidat", conn)
print("=== CANDIDATS ===")
print(df_candidats[['candidat_id', 'nom_candidat', 'niveau_etudes', 'annees_experience', 'statut_candidat']])
print(f"\nNombre de candidats: {len(df_candidats)}")

## 5. Requêtes Débutant (10 questions)

Q1 à Q10 : SELECT, WHERE, LIKE, ORDER BY, COUNT, agrégations, GROUP BY, HAVING, DISTINCT.

In [ ]:
# Q1: Sélectionner tous les candidats
query = "SELECT * FROM profil_candidat"
result = pd.read_sql_query(query, conn)
print(f"Q1: Affiche {len(result)} candidats")
print(result[['candidat_id', 'nom_candidat', 'email']].head(10))

In [ ]:
# Q2: Colonnes spécifiques
query = "SELECT nom_candidat, email FROM profil_candidat"
result = pd.read_sql_query(query, conn)
print(f"Q2: {len(result)} candidats avec nom et email")
print(result.head())

In [ ]:
# Q3: Filtrer avec WHERE
query = "SELECT * FROM offre WHERE salaire_min > 15000"
result = pd.read_sql_query(query, conn)
print(f"Q3: {len(result)} offres avec salaire_min > 15000")
print(result[['offre_id', 'titre_poste', 'salaire_min', 'salaire_max']])

In [ ]:
# Q4: LIKE
query = "SELECT * FROM profil_candidat WHERE nom_candidat LIKE '%Bennani%'"
result = pd.read_sql_query(query, conn)
print(f"Q4: {len(result)} candidats contenant 'Bennani'")
print(result[['nom_candidat', 'email']])

In [ ]:
# Q5: ORDER BY
query = "SELECT nom_entreprise, nombre_employes FROM entreprise ORDER BY nombre_employes DESC"
result = pd.read_sql_query(query, conn)
print(f"Q5: Entreprises triées par nombre d'employés (DESC)")
print(result)

In [ ]:
# Q6: COUNT
query = "SELECT COUNT(*) AS nombre_offres_ouvertes FROM offre WHERE statut = 'Ouvert'"
result = pd.read_sql_query(query, conn)
print(f"Q6: Nombre d'offres ouvertes: {result['nombre_offres_ouvertes'][0]}")

In [ ]:
# Q7: Agrégations multiples
query = """
    SELECT 
        MIN(salaire_min) AS salaire_min_global,
        MAX(salaire_max) AS salaire_max_global,
        ROUND(AVG((salaire_min + salaire_max) / 2), 2) AS salaire_moyen
    FROM offre
"""
result = pd.read_sql_query(query, conn)
print("Q7: Statistiques salariales")
print(result)

In [ ]:
# Q8: GROUP BY
query = """
    SELECT 
        e.secteur,
        COUNT(o.offre_id) AS nombre_offres
    FROM entreprise e
    LEFT JOIN offre o ON e.entreprise_id = o.entreprise_id
    GROUP BY e.secteur
    ORDER BY nombre_offres DESC
"""
result = pd.read_sql_query(query, conn)
print("Q8: Nombre d'offres par secteur")
print(result)

In [ ]:
# Q9: HAVING
query = """
    SELECT 
        niveau_etudes,
        COUNT(*) AS nombre_candidats
    FROM profil_candidat
    GROUP BY niveau_etudes
    HAVING COUNT(*) >= 3
    ORDER BY nombre_candidats DESC
"""
result = pd.read_sql_query(query, conn)
print("Q9: Niveaux d'études avec >= 3 candidats")
print(result)

In [ ]:
# Q10: DISTINCT
query = "SELECT DISTINCT localisation FROM profil_candidat ORDER BY localisation"
result = pd.read_sql_query(query, conn)
print("Q10: Localisations uniques des candidats")
print(result)

## 6. Requêtes Avancées

Sous-requêtes IN, sous-requête scalaire et CASE avec GROUP BY.

In [ ]:
# Avancée 1: IN
query = """
    SELECT nom_candidat, email, localisation
    FROM profil_candidat
    WHERE localisation IN (
        SELECT DISTINCT localisation FROM entreprise
    )
    ORDER BY nom_candidat
"""
result = pd.read_sql_query(query, conn)
print(f"Avancée 1: {len(result)} candidats dans les villes avec entreprises")
print(result[['nom_candidat', 'localisation']].head())

In [ ]:
# Avancée 2: Sous-requête scalaire COUNT
query = """
    SELECT 
        nom_candidat,
        localisation,
        (SELECT COUNT(*) 
         FROM offre o 
         INNER JOIN entreprise e ON o.entreprise_id = e.entreprise_id 
         WHERE e.localisation = profil_candidat.localisation
         AND o.statut = 'Ouvert') AS offres_disponibles
    FROM profil_candidat
    ORDER BY offres_disponibles DESC
"""
result = pd.read_sql_query(query, conn)
print("Avancée 2: Candidats avec offres disponibles par ville")
print(result)

In [ ]:
# Avancée 3: CASE avec GROUP BY
query = """
    SELECT 
        CASE 
            WHEN salaire_min < 12000 THEN 'Petit (< 12k)'
            WHEN salaire_min BETWEEN 12000 AND 20000 THEN 'Moyen (12-20k)'
            ELSE 'Élevé (> 20k)'
        END AS categorie_salaire,
        COUNT(*) AS nombre_offres,
        ROUND(AVG(salaire_min), 2) AS salaire_moyen_min
    FROM offre
    GROUP BY categorie_salaire
    ORDER BY COUNT(*) DESC
"""
result = pd.read_sql_query(query, conn)
print("Avancée 3: Offres par catégorie salariale")
print(result)

## 7. Analyses Bonus

### Analyses supplémentaires utiles

In [ ]:
# Analyse 1: Offres par type de contrat
query = """
    SELECT 
        type_contrat,
        COUNT(*) AS nombre_offres,
        ROUND(AVG(salaire_min), 2) AS salaire_min_moyen,
        ROUND(AVG(salaire_max), 2) AS salaire_max_moyen
    FROM offre
    GROUP BY type_contrat
    ORDER BY nombre_offres DESC
"""
result = pd.read_sql_query(query, conn)
print("Analyse: Offres par type de contrat")
print(result)
print()

In [ ]:
# Analyse 2: Candidats par statut
query = """
    SELECT 
        statut_candidat,
        COUNT(*) AS nombre_candidats,
        ROUND(AVG(annees_experience), 2) AS exp_moyenne
    FROM profil_candidat
    GROUP BY statut_candidat
    ORDER BY nombre_candidats DESC
"""
result = pd.read_sql_query(query, conn)
print("Analyse: Candidats par statut")
print(result)
print()

In [ ]:
# Analyse 3: Candidats par niveau d'études et expérience
query = """
    SELECT 
        niveau_etudes,
        COUNT(*) AS nombre_candidats,
        ROUND(AVG(annees_experience), 1) AS exp_moyenne,
        MIN(annees_experience) AS exp_min,
        MAX(annees_experience) AS exp_max
    FROM profil_candidat
    GROUP BY niveau_etudes
    ORDER BY exp_moyenne DESC
"""
result = pd.read_sql_query(query, conn)
print("Analyse: Candidats par niveau d'études")
print(result)
print()

In [ ]:
# Analyse 4: Résumé global
print("\n" + "="*60)
print("RÉSUMÉ GLOBAL DE LA BASE DE DONNÉES")
print("="*60)

# Statistiques entreprises
query = "SELECT COUNT(*) as total, AVG(nombre_employes) as moyenne_employes FROM entreprise"
result = pd.read_sql_query(query, conn)
print(f"\n📊 ENTREPRISES:")
print(f"  - Total: {result['total'][0]}")
print(f"  - Employés moyens: {result['moyenne_employes'][0]:.0f}")

# Statistiques offres
query = """
    SELECT 
        COUNT(*) as total,
        SUM(CASE WHEN statut = 'Ouvert' THEN 1 ELSE 0 END) as ouvertes,
        ROUND(AVG(salaire_min), 2) as salaire_min_moyen,
        ROUND(AVG(salaire_max), 2) as salaire_max_moyen
    FROM offre
"""
result = pd.read_sql_query(query, conn)
print(f"\n💼 OFFRES:")
print(f"  - Total: {result['total'][0]}")
print(f"  - Ouvertes: {result['ouvertes'][0]}")
print(f"  - Salaire min moyen: {result['salaire_min_moyen'][0]:.2f}")
print(f"  - Salaire max moyen: {result['salaire_max_moyen'][0]:.2f}")

# Statistiques candidats
query = """
    SELECT 
        COUNT(*) as total,
        SUM(CASE WHEN statut_candidat = 'Actif' THEN 1 ELSE 0 END) as actifs,
        ROUND(AVG(annees_experience), 2) as exp_moyenne
    FROM profil_candidat
"""
result = pd.read_sql_query(query, conn)
print(f"\n👥 CANDIDATS:")
print(f"  - Total: {result['total'][0]}")
print(f"  - Actifs: {result['actifs'][0]}")
print(f"  - Expérience moyenne: {result['exp_moyenne'][0]:.1f} ans")

print("\n" + "="*60)

## 8. Exercices (5 questions résolues)

Exercices traités : **1** (SELECT), **2** (INSERT), **7** (DELETE), **9** (SELECT complexe), **10** (TRANSACTION).
Exécutez les cellules dans l'ordre.

### Exercice 1 : SELECT - Candidats par niveau d'études

In [ ]:
# Exercice 1 : candidats actifs triés par niveau d'études (alphabétique) puis expérience décroissante
query = """
    SELECT nom_candidat, niveau_etudes, annees_experience, email
    FROM profil_candidat
    WHERE statut_candidat = 'Actif'
    ORDER BY niveau_etudes ASC, annees_experience DESC
"""
result = pd.read_sql_query(query, conn)
print(f"Exercice 1 : {len(result)} candidats actifs")
print(result)

### Exercice 2 : INSERT - Nouvelle offre pour TechVision Maroc

In [ ]:
# Exercice 2 : insertion de l'offre "Développeur Senior Python" (entreprise_id = 1)
# Garde-fou : on n'insère pas deux fois si la cellule est relancée
deja = cursor.execute(
    "SELECT COUNT(*) FROM offre WHERE entreprise_id = 1 AND titre_poste = 'Développeur Senior Python'"
).fetchone()[0]

if deja == 0:
    cursor.execute("""
        INSERT INTO offre (entreprise_id, titre_poste, description, salaire_min, salaire_max,
                           niveau_experience, competences_requises, date_publication, date_limite,
                           type_contrat, statut)
        VALUES (1, 'Développeur Senior Python',
                'Rejoignez notre équipe pour développer des applications Python innovantes',
                18000, 28000, 'Expert', 'Python, Django, PostgreSQL, Git',
                CURRENT_DATE, DATE('now', '+30 days'), 'CDI', 'Ouvert')
    """)
    conn.commit()
    print(f"✓ Exercice 2 : {cursor.rowcount} enregistrement inséré")
else:
    print("Exercice 2 : l'offre existe déjà, rien à insérer")

print(pd.read_sql_query(
    "SELECT offre_id, entreprise_id, titre_poste, salaire_min, salaire_max, date_publication, date_limite, type_contrat, statut "
    "FROM offre WHERE titre_poste = 'Développeur Senior Python'", conn))

### Exercice 7 : DELETE - Candidats inactifs depuis plus de 6 mois

Date de référence fixée au 20 octobre 2026 : on supprime les inactifs inscrits avant le 20 avril 2026.

In [ ]:
# Exercice 7 : suppression des candidats inactifs inscrits avant le 2026-04-20
print("Avant suppression :")
print(pd.read_sql_query(
    "SELECT candidat_id, nom_candidat, statut_candidat, date_inscription FROM profil_candidat WHERE statut_candidat = 'Inactif'", conn))

cursor.execute("""
    DELETE FROM profil_candidat
    WHERE statut_candidat = 'Inactif'
      AND date_inscription < '2026-04-20'
""")
conn.commit()
print(f"\n✓ Exercice 7 : {cursor.rowcount} enregistrement supprimé")

print("\nAprès suppression :")
print(pd.read_sql_query("SELECT statut_candidat, COUNT(*) AS nombre FROM profil_candidat GROUP BY statut_candidat", conn))

### Exercice 9 : SELECT complexe - Offres ouvertes par ville pour chaque candidat actif

In [ ]:
# Exercice 9 : pour chaque candidat actif, nombre d'offres ouvertes dans sa ville
query = """
    SELECT
        c.nom_candidat,
        c.localisation,
        COUNT(o.offre_id) AS offres_ouvertes
    FROM profil_candidat c
    LEFT JOIN entreprise e ON e.localisation = c.localisation
    LEFT JOIN offre o      ON o.entreprise_id = e.entreprise_id AND o.statut = 'Ouvert'
    WHERE c.statut_candidat = 'Actif'
    GROUP BY c.candidat_id, c.nom_candidat, c.localisation
    ORDER BY offres_ouvertes DESC, c.nom_candidat ASC
"""
result = pd.read_sql_query(query, conn)
print(f"Exercice 9 : {len(result)} candidats actifs")
print(result)

### Exercice 10 : TRANSACTION - Nouvelle entreprise + nouvelle offre

In [ ]:
# Exercice 10 : transaction complète (BEGIN / COMMIT / ROLLBACK en cas d'erreur)
try:
    cursor.execute("BEGIN")

    # 1) Nouvelle entreprise
    cursor.execute("""
        INSERT INTO entreprise (nom_entreprise, secteur, localisation, nombre_employes, date_creation, site_web)
        VALUES ('EduTech Innovation', 'Éducation', 'Casablanca', 75, CURRENT_DATE, 'www.edutech-innovation.ma')
    """)
    nouvelle_entreprise_id = cursor.lastrowid   # clé étrangère de l'offre

    # 2) Nouvelle offre liée à cette entreprise
    cursor.execute("""
        INSERT INTO offre (entreprise_id, titre_poste, salaire_min, salaire_max,
                           niveau_experience, type_contrat, statut)
        VALUES (?, 'Développeur Web Full Stack', 12000, 18000, 'Confirmé', 'CDI', 'Ouvert')
    """, (nouvelle_entreprise_id,))

    conn.commit()
    print(f"✓ Exercice 10 : transaction validée (entreprise_id = {nouvelle_entreprise_id})")

except sqlite3.Error as erreur:
    conn.rollback()
    print(f"✗ Erreur, ROLLBACK effectué : {erreur}")
    print("  (Normal si la cellule est relancée : le nom de l'entreprise doit être unique.)")

print(pd.read_sql_query("""
    SELECT e.entreprise_id, e.nom_entreprise, o.titre_poste, o.salaire_min, o.salaire_max, o.type_contrat, o.statut
    FROM entreprise e JOIN offre o ON o.entreprise_id = e.entreprise_id
    WHERE e.nom_entreprise = 'EduTech Innovation'
""", conn))

## 9. Fermeture de la Base de Données

Fermons la connexion SQLite.

In [ ]:
# Fermer la connexion
conn.close()
print("✓ Connexion fermée")
print("\n✨ Notebook terminé avec succès!")
print("Pour rejouer le notebook : Exécution > Tout exécuter (ou relancez la cellule 1 pour repartir d'une base vide).")